In [ ]:
# Regularisation (Dropout and Data Augmentation) 
# Baseline is Exercise 2 CNN. Models are selected by VALIDATION accuracy only;


import random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms
from IPython.display import display


SEED = 42
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 1e-3
NUM_WORKERS = 0
DATA_DIR = "./data"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

#  Data preparation 
mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)
plain_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])
# Augmentation: small random rotations (+-10 deg) and translations (up to 10%)
aug_transform = transforms.Compose([
    transforms.RandomAffine(degrees=10, translate=(0.10, 0.10)),
    transforms.ToTensor(), transforms.Normalize(mean, std)])

full_train = datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=plain_transform)
test_set = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=plain_transform)

train_size = int(0.8 * len(full_train))
val_size = len(full_train) - train_size
train_set, val_set = random_split(full_train, [train_size, val_size],
                                  generator=torch.Generator().manual_seed(SEED))

# Augmented copy of the SAME training indices; validation/test stay unaugmented
full_train_aug = datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=aug_transform)
train_aug_set = Subset(full_train_aug, train_set.indices)

train_loader = DataLoader(train_set, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
train_eval_loader = DataLoader(train_set, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)  # unaugmented, for fair train accuracy
train_aug_loader = DataLoader(train_aug_set, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_set, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_set, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f"Device: {DEVICE} | Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

In [ ]:
# Model: Exercise 2 CNN with optional dropout before the final layer 
class CNNClassifier(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(dropout), nn.Linear(64 * 8 * 8, 10)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
#  Helpers 
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            if training:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

def train_model(model, train_dl, val_dl, epochs=EPOCHS, lr=LEARNING_RATE):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for epoch in range(epochs):
        start = time.time()
        tl, ta = run_epoch(model, train_dl, criterion, optimizer)
        vl, va = run_epoch(model, val_dl, criterion)
        for k, v in zip(hist, (tl, vl, ta, va)):
            hist[k].append(v)
        print(f"Epoch {epoch+1:02d}/{epochs} | train loss {tl:.4f} acc {ta:.4f} | "
              f"val loss {vl:.4f} acc {va:.4f} | {time.time()-start:.1f}s")
    return model, hist

def evaluate_model(model, loader, label=""):
    loss, acc = run_epoch(model, loader, nn.CrossEntropyLoss())
    if label:
        print(f"{label}: loss={loss:.4f}, accuracy={acc*100:.2f}%")
    return loss, acc

def plot_history(hist, title):
    ep = range(1, len(hist["train_loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(ep, hist["train_loss"], "o-", label="Training")
    ax[0].plot(ep, hist["val_loss"], "o-", label="Validation")
    ax[0].set(title="Loss", xlabel="Epoch", ylabel="Cross-entropy loss"); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(ep, hist["train_acc"], "o-", label="Training")
    ax[1].plot(ep, hist["val_acc"], "o-", label="Validation")
    ax[1].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy"); ax[1].legend(); ax[1].grid(alpha=.3)
    fig.suptitle(title); plt.tight_layout(); plt.show()